In [2]:
import pandas as pd
import os

filepath = "../DATA-HTML-STOCK/NEPSECompany/NEPSECompanyExtractor.csv"
df = pd.read_csv(filepath)
symbols = df['Symbol'].tolist()

stock_ignored = ["CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO", "AMFIPO", "NMLBS", "ILFCPO"]
stock_no_news = ["HATHPO", "BUDBLP", "CFCLPO", "CORBLP", "EBLPO", "HAMROP", "HGIPO", "JFLPO", "JBNLPO", "KADBLP", "KNBLPO", "CEFLPO", "LBLPO", "MFILPO", "MIDBLP", "NBBLPO", "NABBPO", "NABBPO", "NBBPO", "PFLPO", "PRINPO", "PURBLP", "SBBLJP", "SIFCPO", "SILPO", "SMFDBP", "SYFLPO", "TNBLPO", "TDBLPO", "UFLPO", "WDBLPO", "WMBFPO", "HLBSLP"]

def detailed_prediction(score):
    if score >= 0.60:
        return "Strongly Positive"
    elif score >= 0.20:
        return "Positive"
    elif score >= 0.05:
        return "Slightly Positive"
    elif score > -0.05:
        return "Neutral"
    elif score > -0.20:
        return "Slightly Negative"
    elif score > -0.60:
        return "Negative"
    else:
        return "Strongly Negative"

def apply_3day_decay(df):
    df = df.copy()
    df["Decayed_Sentiment"] = 0.0
    t1, t2, t3 = 0.0, 0.0, 0.0
    decayed = []
    for i, row in df.iterrows():
        score = row["Sentiment_Score"]
        if i == 0:
            t1 = score
        else:
            t3 = t2 / 2
            t2 = t1 / 2
            t1 = score
        final_total = t1 + t2 + t3
        decayed.append(final_total)
    df["Decayed_Sentiment"] = decayed
    df["Decayed_Prediction"] = df["Decayed_Sentiment"].apply(detailed_prediction)
    return df

def check_existing_decay(savepath):
    saved_date = None
    old_df = None
    if os.path.exists(savepath):
        old_df = pd.read_csv(savepath)
        saved_date = pd.to_datetime(old_df["Date_Stock"].iloc[-1])
    return saved_date, old_df

for symbol in symbols:
    if symbol in stock_ignored:
        print(f"\n{symbol} is in ignore list!\n")
        continue

    semifinal_path = f"../DATA-HTML-STOCK/SemiFinalDataset/{symbol}.csv"
    if not os.path.exists(semifinal_path):
        print(f"no semifinal data for {symbol} found, skipping")
        continue

    semifinal_df = pd.read_csv(semifinal_path)

    savepath = f"../DATA-HTML-STOCK/FinalDataSet/{symbol}.csv"
    saved_date, old_df = check_existing_decay(savepath)

    if saved_date is not None:
        new_rows = semifinal_df[pd.to_datetime(semifinal_df["Date_Stock"]) > saved_date]
        if new_rows.empty:
            print(f"{symbol} no new data for decay")
            continue

        last_t1 = old_df["Decayed_Sentiment"].iloc[-1]
        last_t2 = old_df["Decayed_Sentiment"].iloc[-2] if len(old_df) > 1 else 0.0
        new_rows = new_rows.reset_index(drop=True)
        t1, t2, t3 = last_t1, last_t2, 0.0
        decayed = []
        for i, row in new_rows.iterrows():
            score = row["Sentiment_Score"]
            t3 = t2 / 2
            t2 = t1 / 2
            t1 = score
            decayed.append(t1 + t2 + t3)
        new_rows["Decayed_Sentiment"] = decayed
        new_rows["Decayed_Prediction"] = new_rows["Decayed_Sentiment"].apply(detailed_prediction)
        final_df = pd.concat([old_df, new_rows], ignore_index=True)
    else:
        if symbol in stock_no_news:
            print(f"\n{symbol} is in stock no news list!\n")
            semifinal_df["Decayed_Sentiment"] = 0.0
            semifinal_df["Decayed_Prediction"] = "Neutral"
            final_df = semifinal_df
        else:
            final_df = apply_3day_decay(semifinal_df)

    print(final_df)
    final_df.to_csv(savepath, index=False)

      Date_Stock  Close Date_News  Sentiment_Score Prediction  \
0     2026-03-03  305.0       NaN              0.0    Neutral   
1     2026-03-01  302.6       NaN              0.0    Neutral   
2     2026-02-26  298.0       NaN              0.0    Neutral   
3     2026-02-25  294.6       NaN              0.0    Neutral   
4     2026-02-24  293.0       NaN              0.0    Neutral   
...          ...    ...       ...              ...        ...   
3523  2010-09-12  118.0       NaN              0.0    Neutral   
3524  2010-09-09  122.0       NaN              0.0    Neutral   
3525  2010-09-08  125.0       NaN              0.0    Neutral   
3526  2010-09-07  138.0       NaN              0.0    Neutral   
3527  2010-09-02  255.0       NaN              0.0    Neutral   

      Decayed_Sentiment Decayed_Prediction  
0                   0.0            Neutral  
1                   0.0            Neutral  
2                   0.0            Neutral  
3                   0.0            Neut